# Meshtasticator - Mesh Network Simulator (Colab Edition)

This notebook allows you to run the Meshtasticator mesh network simulator on Google Colab or similar cloud platforms.

## Features:
- ✅ Automated environment setup
- ✅ Checkpoint support for long-running simulations
- ✅ **Automatic real-time backup to Google Drive after EACH run** 🔄
- ✅ Resume capability if session disconnects
- ✅ Visualization support

## Setup Time: ~2-3 minutes

## Step 1: Mount Google Drive (HIGHLY RECOMMENDED)

Mount your Google Drive to enable automatic backup after each run. **This is the safest option!**

With auto-sync enabled:
- ✅ Results backed up after EVERY completed run
- ✅ Zero data loss if Colab disconnects
- ✅ Can resume from any point

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a directory for your project results
project_dir = '/content/drive/MyDrive/Meshtasticator_Results'
os.makedirs(project_dir, exist_ok=True)
print(f"✅ Results will be automatically synced to: {project_dir}")
print(f"✅ Auto-sync ENABLED - results backed up after each run!")

## Step 2: Clone Repository

Clone the Meshtasticator repository from GitHub.

In [ ]:
import os

# Remove existing directory if it exists
if os.path.exists('/content/Meshtasticator'):
    !rm -rf /content/Meshtasticator

# Clone your repository (replace with your actual repository URL)
!git clone https://github.com/YOUR_USERNAME/Meshtasticator.git /content/Meshtasticator

# Change to project directory
%cd /content/Meshtasticator

print("\n✅ Repository cloned successfully!")

## Step 3: Install Dependencies

Install all required Python packages.

In [ ]:
# Install requirements
!pip install -q -r requirements.txt

print("✅ Dependencies installed successfully!")

# Verify critical imports
try:
    import simpy
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
    import yaml
    print("✅ All critical packages verified!")
except ImportError as e:
    print(f"❌ Import error: {e}")

## Step 4: Configure Simulation Parameters

View and optionally modify your topology parameters.

In [ ]:
# Display current topology parameters
!cat topology_params.yaml

## Step 5: Run Batch Simulations (WITH AUTO-SYNC TO DRIVE)

Run multiple simulations with:
- ✅ **Automatic Google Drive backup after EACH completed run**
- ✅ Checkpoint support
- ✅ Can resume if disconnected

**How it works:**
1. Each run completes
2. Results immediately synced to Google Drive
3. Checkpoint saved
4. Next run starts

**If Colab disconnects**: Just resume and continue - no data loss! 🎉

In [ ]:
# Configuration
NUM_RUNS = 10  # Number of simulation runs
ROUTING_TYPES = ['AODV', 'MANAGED_FLOOD', 'BL_A_AODV']  # Routing protocols to test

# Run batch simulations with CHECKPOINT + AUTO-SYNC TO DRIVE
!python run_batch_simulations.py \
    --runs {NUM_RUNS} \
    --routing {' '.join(ROUTING_TYPES)} \
    --checkpoint \
    --auto-sync

print("\n" + "="*70)
print("💾 Results continuously backed up to Google Drive after each run!")
print(f"📁 Location: {project_dir}")
print("="*70)

## Step 6: Resume from Checkpoint (If Session Disconnected)

If your session was interrupted, simply resume! Your progress is safely backed up in Google Drive.

In [ ]:
# If you need to resume after disconnection:
# 1. Re-run Steps 1-3 (mount drive, clone repo, install deps)
# 2. List available checkpoints
!ls -ltr batch_results_*/ 2>/dev/null | tail -5

# 3. Resume from the last batch (replace with your actual directory name)
CHECKPOINT_DIR = "batch_results_20260214_120000"  # Replace with your directory

# Resume with auto-sync enabled
!python run_batch_simulations.py \
    --resume {CHECKPOINT_DIR} \
    --auto-sync

print("\n✅ Resumed and continuing to sync to Google Drive!")

## Step 7: Check Sync Status

Verify that your results are safely backed up in Google Drive.

In [ ]:
from pathlib import Path
import json

# Check local results
local_batches = list(Path('.').glob('batch_results_*'))
print(f"📊 Local batch directories: {len(local_batches)}")

# Check Google Drive backups
if 'project_dir' in globals():
    drive_batches = list(Path(project_dir).glob('batch_results_*'))
    print(f"☁️  Google Drive backups: {len(drive_batches)}")
    
    # Show most recent
    if drive_batches:
        latest = sorted(drive_batches)[-1]
        print(f"\n✅ Latest backup: {latest}")
        
        # Check checkpoint
        checkpoint_file = latest / 'checkpoint.json'
        if checkpoint_file.exists():
            with open(checkpoint_file, 'r') as f:
                checkpoint = json.load(f)
                print(f"   Progress: {checkpoint['progress_percentage']:.1f}% complete")
                print(f"   Completed: {len(checkpoint['completed_runs'])}/{checkpoint['total_runs']} runs")
else:
    print("⚠️  Google Drive not mounted")

## Step 8: Analyze Results

View and analyze the simulation results.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Find the most recent batch results
batch_dirs = sorted(Path('.').glob('batch_results_*'))
if batch_dirs:
    latest_batch = batch_dirs[-1]
    print(f"Latest batch: {latest_batch}")
    
    # Display summary
    summary_file = latest_batch / 'BATCH_SUMMARY.txt'
    if summary_file.exists():
        with open(summary_file, 'r') as f:
            print(f.read())
    
    # Load JSON summary
    json_summary = latest_batch / 'batch_summary.json'
    if json_summary.exists():
        with open(json_summary, 'r') as f:
            data = json.load(f)
            print(f"\n📊 Batch Statistics:")
            print(f"   Total Runs: {data['total_runs']}")
            print(f"   Successful: {data['successful_runs']}")
            print(f"   Failed: {data['failed_runs']}")
            print(f"   Duration: {data['duration_minutes']:.2f} minutes")
else:
    print("No batch results found. Run simulations first!")

## Step 9: Visualize Results

Generate plots and visualizations from the simulation data.

In [ ]:
# Run analysis scripts
if batch_dirs:
    latest_batch = batch_dirs[-1]
    
    # Analyze broadcast comparison
    !python plot_broadcast_interval_comparison.py
    
    # Compare routing protocols
    !python compare_reliability.py
    !python compare_delays.py
    
    print("\n✅ Analysis complete! Check the output folders for plots.")

## Step 10: Download Results (Optional)

If you prefer to download results directly to your computer (in addition to Google Drive backup).

In [ ]:
# Create a zip archive of all results
import shutil
from google.colab import files

if batch_dirs:
    latest_batch = batch_dirs[-1]
    
    # Create zip file
    zip_name = f"{latest_batch.name}_results"
    shutil.make_archive(zip_name, 'zip', latest_batch)
    
    # Download
    print(f"Downloading {zip_name}.zip...")
    files.download(f"{zip_name}.zip")
    print("✅ Download started!")
else:
    print("No results to download.")

## 💡 How Auto-Sync Works

### Traditional Approach (Manual):
```
Run 1 → Run 2 → ... → Run 10 → [Manual Copy to Drive]
```
❌ If disconnection happens at Run 7, lose all progress

### New Auto-Sync Approach:
```
Run 1 → [Sync] → Run 2 → [Sync] → Run 3 → [Sync] ...
```
✅ Each run immediately backed up
✅ If disconnection happens at Run 7, can resume from Run 8
✅ Zero data loss

### What Gets Synced After Each Run:
- All simulation outputs (CSV, PNG, PKL files)
- Checkpoint files
- Configuration files
- Metadata

### Efficiency:
- Only changed/new files are copied (incremental sync)
- Fast and network-efficient
- No impact on simulation performance

## Tips for Long-Running Simulations

1. **Always use `--auto-sync`** when running on Colab with Drive mounted
2. **Keep browser tab active** to prevent disconnection
3. **Use Colab Pro** for longer runtime limits (24h vs 2-4h)
4. **Check sync status** periodically to ensure backups are working
5. **Resume capability**: If disconnected, just re-run Steps 1-3, then Step 6

## Troubleshooting

**Sync not working?**
- Ensure Google Drive is mounted (Step 1)
- Check that `project_dir` variable is set
- Verify Drive has enough space

**Session Disconnected?**
1. Re-run Step 1 (Mount Drive)
2. Re-run Step 2 (Clone Repository) 
3. Re-run Step 3 (Install Dependencies)
4. Go to Step 6 (Resume from Checkpoint)

**Out of Memory?**
- Reduce `NUM_RUNS`
- Reduce number of nodes in `topology_params.yaml`
- Use fewer routing protocols

**Simulation Too Slow?**
- Reduce simulation time in config files
- Use smaller network topologies
- Consider Colab Pro for better resources